# ConfidentAI: Personal Confidence Booster — Exploratory Data Analysis & Machine Learning Pipeline
---
### Academic Capstone ML Project
**Objective:** Estimate interpersonal communication and confidence from self-assessments, acoustic speech delivery metrics, and linguistic NLP features.
**Pipeline:** Data Generation & Ingestion -> Preprocessing -> Feature Engineering -> Multi-Model Training -> Evaluation -> Explainability.

In [1]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print('Libraries loaded successfully!')

## 1. Load Dataset & Summary Statistics

In [2]:
data_path = os.path.join('..', 'data', 'confidence_dataset.csv')
df = pd.read_csv(data_path)
print(f'Dataset shape: {df.shape[0]} samples, {df.shape[1]} columns')
df.head()

In [3]:
print('Target Category Distribution:')
print(df['confidence_category'].value_counts())
print('\nSummary Statistics:')
df.describe().round(2)

## 2. Feature Engineering
We construct domain composite indices:
1. **Self-Efficacy Index**: Normalized combination of positive Likert indicators.
2. **Anxiety Penalty Index**: Normalized combination of hesitation, nervousness, and fear of judgment.
3. **Pacing Optimality**: Gaussian curve centered on ideal conversational rate (145 WPM).
4. **Fluency Composite**: Pacing, filler word density, pause ratio, and speech stability.

In [4]:
def add_features(data):
    d = data.copy()
    pos = (d['public_speaking_comfort'] + d['interview_confidence'] + d['opinion_expression_comfort'] + d['unexpected_questions_comfort'] + d['technical_explanation_comfort']) / 5.0
    d['self_efficacy_index'] = ((pos - 1.0) / 4.0).clip(0, 1)
    neg = (d['hesitation_frequency'] + d['nervousness_level'] + d['fear_of_judgment']) / 3.0
    d['anxiety_penalty_index'] = ((neg - 1.0) / 4.0).clip(0, 1)
    wpm_diff = np.abs(d['words_per_minute'] - 145.0)
    d['pacing_optimality'] = np.exp(-0.5 * (wpm_diff / 35.0)**2).clip(0, 1)
    d['filler_density'] = (d['filler_word_count'] / np.maximum(d['response_length_words'], 10) * 100.0).clip(0, 30)
    d['pause_ratio'] = ((d['pause_frequency'] * (d['speaking_duration'] / 60.0) * 0.8) / np.maximum(d['speaking_duration'], 5.0)).clip(0, 0.85)
    d['fluency_composite'] = (0.35 * d['pacing_optimality'] + 0.30 * np.maximum(0, 1 - d['filler_density']/12.0) + 0.20 * np.maximum(0, 1 - d['pause_ratio']/0.5) + 0.15 * d['speech_consistency']).clip(0, 1)
    d['clarity_sentiment_interaction'] = ((d['clarity_score']/100.0) * ((d['sentiment_score']+1.0)/2.0)).clip(0, 1)
    return d

df_feat = add_features(df)
print('Engineered Features Created. Total Columns:', len(df_feat.columns))

## 3. Dataset Preprocessing & Splitting

In [5]:
feature_cols = [c for c in df_feat.columns if c not in ['confidence_score', 'confidence_category']]
X = df_feat[feature_cols]
y_reg = df_feat['confidence_score']
y_clf_raw = df_feat['confidence_category']

le = LabelEncoder()
le.fit(['Low', 'Moderate', 'High'])
y_clf = le.transform(y_clf_raw)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train size: {len(X_train_scaled)}, Test size: {len(X_test_scaled)}')

## 4. Multi-Model Regression Benchmarking

In [6]:
reg_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
}

print(f"{'Algorithm':<26} {'MAE':<10} {'RMSE':<10} {'R2 Score':<10}")
print('-' * 56)
for name, model in reg_models.items():
    model.fit(X_train_scaled, y_reg_train)
    preds = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_reg_test, preds)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    r2 = r2_score(y_reg_test, preds)
    print(f"{name:<26} {mae:<10.3f} {rmse:<10.3f} {r2:<10.4f}")

## 5. Multi-Model Classification Benchmarking

In [7]:
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, random_state=42),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
}

print(f"{'Algorithm':<26} {'Accuracy':<10} {'F1-Score':<10}")
print('-' * 46)
for name, model in clf_models.items():
    model.fit(X_train_scaled, y_clf_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_clf_test, preds)
    f1 = f1_score(y_clf_test, preds, average='weighted')
    print(f"{name:<26} {acc*100:<9.2f}% {f1:<10.4f}")

## 6. Feature Importance Attribution (Random Forest)

In [8]:
rf = reg_models['Random Forest Regressor']
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('Top 8 Most Influential Features:')
print((importances.head(8) * 100).round(2))
print('\nConfidence Analysis Notebook Ready for Viva Defense!')